In [1]:
import joblib
import pandas as pd
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
import warnings

In [2]:
target_column = "health_condition"

In [3]:
X_train = pd.read_csv("../data/intermediate/train_features.csv")
X_valid = pd.read_csv("../data/intermediate/valid_features.csv")
X_test = pd.read_csv("../data/intermediate/test_features.csv")

y_train = pd.read_csv("../data/intermediate/train_labels.csv")
y_valid = pd.read_csv("../data/intermediate/valid_labels.csv")

X_train['diet_type'] = X_train['diet_type'].astype('category')
X_valid['diet_type'] = X_valid['diet_type'].astype('category')
X_test['diet_type'] = X_test['diet_type'].astype('category')

X_train['gender'] = X_train['gender'].astype('category')
X_valid['gender'] = X_valid['gender'].astype('category')
X_test['gender'] = X_test['gender'].astype('category')

X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 552070 entries, 0 to 552069
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   sleep_duration                491266 non-null  float64 
 1   heart_rate                    545802 non-null  float64 
 2   bmi                           541053 non-null  float64 
 3   calorie_expenditure           509705 non-null  float64 
 4   step_count                    540909 non-null  float64 
 5   exercise_duration             546550 non-null  float64 
 6   water_intake                  517224 non-null  float64 
 7   diet_type                     546580 non-null  category
 8   stress_level                  485727 non-null  float64 
 9   sleep_quality                 505469 non-null  float64 
 10  physical_activity_level       522744 non-null  float64 
 11  smoking_alcohol               529184 non-null  float64 
 12  gender                        535022 non-

In [4]:
X = pd.concat([X_train, X_valid], axis=0)
y = pd.concat([y_train, y_valid], axis=0)

In [5]:
df_submission = pd.read_csv("../data/sample_submission.csv")
df_submission.info()

<class 'pandas.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                295753 non-null  int64
 1   health_condition  295753 non-null  str  
dtypes: int64(1), str(1)
memory usage: 4.5 MB


In [6]:
train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
valid_sample_weight = compute_sample_weight(class_weight="balanced", y=y_valid)
y_sample_weight = compute_sample_weight(class_weight="balanced", y=y)

binary_models = []

for i in range(3):
    y_train_binary = y_train == i
    y_valid_binary = y_valid == i

    model = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=8, objective='binary', early_stopping_round=100)
    model.fit(X_train, y_train_binary, sample_weight=train_sample_weight, eval_set=[(X_valid, y_valid_binary)])

    binary_models.append(model)

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\ligh

[1]	valid_0's binary_logloss: 0.412444
[2]	valid_0's binary_logloss: 0.370879
[3]	valid_0's binary_logloss: 0.335925
[4]	valid_0's binary_logloss: 0.306133
[5]	valid_0's binary_logloss: 0.280712
[6]	valid_0's binary_logloss: 0.258614
[7]	valid_0's binary_logloss: 0.239547
[8]	valid_0's binary_logloss: 0.222877
[9]	valid_0's binary_logloss: 0.208328
[10]	valid_0's binary_logloss: 0.195584
[11]	valid_0's binary_logloss: 0.184362
[12]	valid_0's binary_logloss: 0.174535
[13]	valid_0's binary_logloss: 0.165825
[14]	valid_0's binary_logloss: 0.158186
[15]	valid_0's binary_logloss: 0.151369
[16]	valid_0's binary_logloss: 0.145399
[17]	valid_0's binary_logloss: 0.14007
[18]	valid_0's binary_logloss: 0.135361
[19]	valid_0's binary_logloss: 0.131207
[20]	valid_0's binary_logloss: 0.127491
[21]	valid_0's binary_logloss: 0.124184
[22]	valid_0's binary_logloss: 0.121233
[23]	valid_0's binary_logloss: 0.11862
[24]	valid_0's binary_logloss: 0.116277
[25]	valid_0's binary_logloss: 0.11422
[26]	valid_0

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\ligh

[1]	valid_0's binary_logloss: 0.863017
[2]	valid_0's binary_logloss: 0.758975
[3]	valid_0's binary_logloss: 0.676874
[4]	valid_0's binary_logloss: 0.610194
[5]	valid_0's binary_logloss: 0.555111
[6]	valid_0's binary_logloss: 0.508914
[7]	valid_0's binary_logloss: 0.469633
[8]	valid_0's binary_logloss: 0.435924
[9]	valid_0's binary_logloss: 0.406854
[10]	valid_0's binary_logloss: 0.381652
[11]	valid_0's binary_logloss: 0.359814
[12]	valid_0's binary_logloss: 0.34066
[13]	valid_0's binary_logloss: 0.323829
[14]	valid_0's binary_logloss: 0.309002
[15]	valid_0's binary_logloss: 0.295956
[16]	valid_0's binary_logloss: 0.284313
[17]	valid_0's binary_logloss: 0.274108
[18]	valid_0's binary_logloss: 0.264983
[19]	valid_0's binary_logloss: 0.256848
[20]	valid_0's binary_logloss: 0.249623
[21]	valid_0's binary_logloss: 0.2432
[22]	valid_0's binary_logloss: 0.237333
[23]	valid_0's binary_logloss: 0.232194
[24]	valid_0's binary_logloss: 0.227538
[25]	valid_0's binary_logloss: 0.223398
[26]	valid_0

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\ligh

[1]	valid_0's binary_logloss: 0.39816
[2]	valid_0's binary_logloss: 0.358704
[3]	valid_0's binary_logloss: 0.325168
[4]	valid_0's binary_logloss: 0.29669
[5]	valid_0's binary_logloss: 0.272074
[6]	valid_0's binary_logloss: 0.250437
[7]	valid_0's binary_logloss: 0.231542
[8]	valid_0's binary_logloss: 0.215006
[9]	valid_0's binary_logloss: 0.200481
[10]	valid_0's binary_logloss: 0.187936
[11]	valid_0's binary_logloss: 0.176702
[12]	valid_0's binary_logloss: 0.166779
[13]	valid_0's binary_logloss: 0.157982
[14]	valid_0's binary_logloss: 0.150316
[15]	valid_0's binary_logloss: 0.143546
[16]	valid_0's binary_logloss: 0.137556
[17]	valid_0's binary_logloss: 0.132259
[18]	valid_0's binary_logloss: 0.127504
[19]	valid_0's binary_logloss: 0.123284
[20]	valid_0's binary_logloss: 0.119557
[21]	valid_0's binary_logloss: 0.116203
[22]	valid_0's binary_logloss: 0.113255
[23]	valid_0's binary_logloss: 0.110682
[24]	valid_0's binary_logloss: 0.108397
[25]	valid_0's binary_logloss: 0.106293
[26]	valid_

In [7]:
from sklearn.metrics import balanced_accuracy_score

y_pred_eq0 = binary_models[0].predict(X_valid)
y_pred_eq1 = binary_models[1].predict(X_valid)
y_pred_eq2 = binary_models[2].predict(X_valid)

y_pred = np.clip(y_pred_eq1 + (y_pred_eq2 * 2), 0, 2)

val_score = balanced_accuracy_score(y_valid, y_pred, sample_weight=valid_sample_weight)
print("Validation Balanced Accuracy:", val_score)

Validation Balanced Accuracy: 0.9494153157252209


In [21]:
model = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=8, objective='multiclass', enable_categorical=True)
model.fit(X, y, sample_weight=y_sample_weight, categorical_feature="name:diet_type,gender")

joblib.dump(model, '../models/lgbm_100.pkl')

[LightGBM] [Warning] Unknown parameter: enable_categorical

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)



[LightGBM] [Warning] Unknown parameter: enable_categorical
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013444 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2831
[LightGBM] [Info] Number of data points in the train set: 690088, number of used features: 18
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


['../models/lgbm_100.pkl']

In [22]:
y_pred = model.predict(X_test)

df_submission[target_column] = y_pred
df_submission[target_column] = df_submission[target_column].replace({0:'unhealthy', 1:'at-risk', 2: 'fit'})

df_submission.to_csv('../results/lgbm_baseline.csv', index=False)
df_submission

[LightGBM] [Warning] Unknown parameter: enable_categorical


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
...,...,...
295748,985836,fit
295749,985837,at-risk
295750,985838,unhealthy
295751,985839,at-risk
